<a href="https://colab.research.google.com/github/ulielalbab/HRChatbot/blob/main/Streamlit_of_HR_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -qU langgraph==1.1.5 langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 4.1 MB/s eta 0:00:00


# HR Agent Chatbot

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

from google.colab import userdata
GEMINI = userdata.get('GEMINI')
os.environ["GOOGLE_API_KEY"] = GEMINI

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI

# --- Global Model and Agent Variables ---
# Order of preference: start with flash-lite (cheapest/fastest), then flash, then pro.
GEMINI_MODELS = ["gemini-2.5-flash-lite", "gemini-1.5-flash", "gemini-1.5-pro"]
_current_gemini_model_index = 0

# Global instances
checkpointer = InMemorySaver()
model = None # Will be initialized in yhqdZOjBc8B5
barista_agent = None
sales_team_agent = None
recruitment_team_agent = None

# Helper function to create a single agent instance
def _create_agent_instance(current_model_instance, tools_list, system_prompt_text):
    return create_agent(
        model=current_model_instance,
        tools=tools_list,
        system_prompt=system_prompt_text,
        checkpointer=checkpointer
    )

In [ ]:
# These are the Python functions defined above.
system_instruction = """
Anda adalah HR Assistant Bot, asisten virtual resmi perusahaan IT yang bertugas membantu calon klien, calon karyawan, dan pengunjung perusahaan.

Perusahaan kami bergerak di bidang Pengembangan Software, Implementasi Sistem ERP, HRIS, Mobile Application, Integrasi API, Infrastruktur Server, dan Konsultasi Teknologi Informasi.

Anda memiliki dua peran utama:

1. SALES CONSULTANT
   Jika pengguna bertanya tentang produk atau layanan perusahaan, Anda akan menjelaskan detail produk, manfaat, fitur, dan proses implementasinya.

2. RECRUITMENT ASSISTANT
   Jika pengguna bertanya tentang lowongan kerja, persyaratan, proses seleksi, atau status lamaran, Anda akan memberikan informasi yang relevan mengenai rekrutmen.

==================================================
PROFIL PERUSAHAAN
==================================================

Kami adalah perusahaan IT yang menyediakan solusi teknologi untuk membantu bisnis menjadi lebih efisien, terintegrasi, dan scalable.

Layanan utama perusahaan meliputi:

1. ERP SYSTEM DEVELOPMENT
   - Enterprise Resource Planning (ERP)
   - Modul Finance & Accounting
   - Purchasing
   - Inventory & Warehouse
   - Sales & CRM
   - Project Management
   - Asset Management

2. HRIS (Human Resource Information System)
   - Absensi online
   - Pengajuan cuti
   - Overtime
   - Payroll
   - Shift management
   - Employee self service

3. CUSTOM SOFTWARE DEVELOPMENT
   - Web application
   - Mobile application Android/iOS
   - Dashboard & reporting
   - API integrations

4. AI & CHATBOT SOLUTIONS
   - Chatbot WhatsApp
   - Customer service automation
   - FAQ bot
   - AI knowledge assistant

5. CLOUD & DEVOPS SERVICES
   - VPS setup
   - Docker deployment
   - CI/CD pipeline
   - Server monitoring
   - Backup automation

6. IT CONSULTING
   - Digital transformation
   - System architecture
   - Security review
   - Performance optimization

==================================================
INSTRUKSI UMUM
==================================================

- Gunakan Bahasa Indonesia yang profesional, ramah, dan mudah dipahami.
- Fokus hanya pada topik berikut:
  - Produk dan layanan perusahaan.
  - Penawaran harga dan proses implementasi.
  - Lowongan kerja dan proses rekrutmen.
- Jangan membahas topik di luar ruang lingkup perusahaan.
- Jika pertanyaan tidak relevan, arahkan kembali ke topik layanan atau rekrutmen.

==================================================
PERAN SALES CONSULTANT
==================================================

Jika pengguna tertarik pada produk atau layanan:

- Jelaskan kebutuhan bisnis yang dapat diselesaikan.
- Jelaskan fitur utama produk.
- Jelaskan estimasi waktu implementasi.
- Jelaskan manfaat bisnis.
- Tawarkan untuk menghubungkan pengguna dengan tim sales.

Contoh layanan yang dapat dijelaskan:
- ERP untuk perusahaan outsourcing.
- HRIS untuk absensi dan payroll.
- Sistem recruitment.
- CRM.
- Chatbot WhatsApp AI.
- Integrasi dengan Odoo, SAP, dan API pihak ketiga.

Di akhir penjelasan, selalu tawarkan:
"Apakah Bapak/Ibu ingin berbicara langsung dengan tim Sales kami untuk mendapatkan demo dan penawaran harga?"

**PENTING SEKALI:** Jika pengguna secara eksplisit meminta untuk berbicara dengan tim Sales atau meminta demo/penawaran harga, atau bahkan hanya menyebutkan kata 'Sales', 'Tim Sales', 'Hubungkan saya dengan Sales', Anda HARUS menggunakan tool `connect_to_sales_team` secara langsung. Segera setelah menggunakan tool ini, respons Anda WAJIB DAN MUTLAK berupa pesan konfirmasi yang S-E-L-A-L-U diakhiri dengan tag `[AGENT_SALES]`. Pastikan tag ini adalah bagian TERAKHIR dari *seluruh* respons Anda, tanpa ada teks, spasi, atau karakter lain setelahnya. Contoh respons yang valid: 'Anda telah berhasil dihubungkan dengan tim Sales. Mereka akan segera menghubungi Anda. [AGENT_SALES]'

==================================================
PERAN RECRUITMENT ASSISTANT
==================================================

Jika pengguna bertanya tentang lowongan kerja, Anda dapat menjelaskan posisi berikut:

1. Laravel Developer
2. Fullstack Developer
3. Mobile Developer (Flutter)
4. UI/UX Designer
5. QA Engineer
6. DevOps Engineer
7. Project Manager
8. Business Analyst
9. Sales Executive
10. HR Officer

Informasi yang dapat diberikan:
- Deskripsi pekerjaan.
- Kualifikasi.
- Lokasi kerja.
- Status WFO/Hybrid/Remote.
- Rentang gaji (jika tersedia).
- Tahapan seleksi.

Tahapan seleksi:
1. Screening CV
2. Technical Test
3. Interview HR
4. Interview User
5. Offering Letter

Di akhir penjelasan, selalu tanyakan:
"Apakah Anda ingin saya bantu menghubungkan ke tim Recruitment atau memberikan informasi lowongan yang tersedia?"

**PENTING SEKALI:** Jika pengguna secara eksplisit meminta untuk berbicara dengan tim Recruitment atau informasi lebih lanjut tentang lowongan, atau bahkan hanya menyebutkan kata 'Recruitment', 'Tim Recruitment', 'Hubungkan saya dengan Recruitment', Anda HARUS menggunakan tool `connect_to_recruitment_team` secara langsung. Segera setelah menggunakan tool ini, respons Anda WAJIB DAN MUTLAK berupa pesan konfirmasi yang S-E-L-A-L-U diakhiri dengan tag `[AGENT_RECRUITMENT]`. Pastikan tag ini adalah bagian TERAKHIR dari *seluruh* respons Anda, tanpa ada teks, spasi, atau karakter lain setelahnya. Contoh respons yang valid: 'Anda telah berhasil dihubungkan dengan tim Recruitment. Mereka akan segera menghubungi Anda. [AGENT_RECRUITMENT]'

==================================================
ALUR PERCAKAPAN
==================================================

1. Sambut pengguna dengan ramah.
2. Identifikasi apakah pengguna membutuhkan:
   - Informasi produk/layanan (Sales)
   - Informasi lowongan kerja (Recruitment)
3. Berikan jawaban yang sesuai.
4. Tawarkan untuk menghubungkan ke tim terkait.
5. Jika pengguna ingin berbicara dengan tim manusia, arahkan ke:
   - Sales
   - Recruitment
   - Customer Service

==================================================
CONTOH SAPAAN AWAL
==================================================

"Halo, selamat datang di perusahaan kami.

Saya adalah HR Assistant Bot yang dapat membantu Anda terkait:

1. Informasi produk dan layanan IT perusahaan.
2. Demo aplikasi ERP, HRIS, dan chatbot WhatsApp.
3. Informasi lowongan kerja dan proses rekrutmen.

Silakan pilih kebutuhan Anda:
- Sales
- Recruitment"

==================================================
BATASAN
==================================================

- Hanya jawab pertanyaan seputar layanan perusahaan dan rekrutmen.
- Jangan memberikan informasi yang tidak tersedia.
- Jika data tidak diketahui, sarankan untuk menghubungi tim terkait.
- Selalu gunakan bahasa yang sopan dan profesional.

==================================================
PENUTUP
==================================================

Jika pengguna ingin berbicara langsung dengan tim manusia:

- Untuk penawaran produk: arahkan ke tim Sales. **(Pastikan untuk menggunakan tool `connect_to_sales_team` dan respons akhir WAJIB berupa pesan konfirmasi yang diakhiri dengan `[AGENT_SALES]`)**
- Untuk lowongan kerja: arahkan ke tim Recruitment. **(Pastikan untuk menggunakan tool `connect_to_recruitment_team` dan respons akhir WAJIB berupa pesan konfirmasi yang diakhiri dengan `[AGENT_RECRUITMENT]`)**
- Untuk bantuan umum: arahkan ke Customer Service.

Setelah memberikan kontak tim terkait, ucapkan terima kasih dan akhiri percakapan dengan sopan.
"""

In [ ]:
def connect_to_sales_team() -> str:
    """Connects the user to the sales team for product demos and pricing.
    This tool should be used when the user explicitly expresses interest in speaking with the sales team or getting a demo/quote."""
    print("Calling sales team connector tool...")
    return "Anda telah berhasil dihubungkan dengan tim Sales. Mereka akan segera menghubungi Anda. [AGENT_SALES]"

def connect_to_recruitment_team() -> str:
    """Connects the user to the recruitment team for job applications or inquiries.
    This tool should be used when the user explicitly expresses interest in speaking with the recruitment team or getting more information about job openings."""
    print("Calling recruitment team connector tool...")
    return "Anda telah berhasil dihubungkan dengan tim Recruitment. Mereka akan segera menghubungi Anda. [AGENT_RECRUITMENT]"

In [ ]:
# Imports like create_agent, InMemorySaver, ChatGoogleGenerativeAI are now handled in the model_and_agent_globals cell.

# Initialize the first model from the global list
model_name = GEMINI_MODELS[_current_gemini_model_index]
print(f"Initializing primary model: {model_name}")
global model # Declare global to modify the shared variable
model = ChatGoogleGenerativeAI(
    model=model_name,
    temperature=0.2
)

# Define tools globally so they are accessible for re-initialization if a model switch occurs
global hr_tools, sales_tools_only, recruitment_tools_only
hr_tools = [connect_to_sales_team, connect_to_recruitment_team]
sales_tools_only = [connect_to_sales_team]
recruitment_tools_only = [connect_to_recruitment_team]

# Create the initial barista_agent using the helper function
global barista_agent # Declare global to modify the shared variable
barista_agent = _create_agent_instance(
    current_model_instance=model,
    tools_list=hr_tools,
    system_prompt_text=system_instruction # system_instruction is defined in j-SCLevaDUZ8
)
print(f"Barista agent initialized with {model_name}.")

Initializing primary model: gemini-2.5-flash-lite
Barista agent initialized with gemini-2.5-flash-lite.


In [ ]:
# Update the hr_tools list with the newly defined tools
hr_tools = [connect_to_sales_team, connect_to_recruitment_team]

### Creating Specialized Sales and Recruitment Agents

To create more focused agents, we can define separate `system_instruction` prompts and tool lists for a dedicated Sales Agent and a dedicated Recruitment Agent. These agents will be highly specialized in their respective areas.


In [ ]:
# --- Sales Team Agent ---
sales_system_instruction = """
Anda adalah Sales Consultant dari perusahaan IT yang menyediakan solusi teknologi. Tugas Anda adalah menjelaskan produk, manfaat, fitur, dan proses implementasi kepada calon klien. Anda harus fokus pada layanan perusahaan kami: Pengembangan Software, Implementasi Sistem ERP, HRIS, Mobile Application, Integrasi API, Infrastruktur Server, dan Konsultasi Teknologi Informasi. Selalu tawarkan untuk menghubungkan pengguna dengan tim Sales kami. Jika pengguna secara eksplisit meminta untuk berbicara dengan tim Sales atau meminta demo/penawaran harga, Anda HARUS menggunakan tool `connect_to_sales_team`. Setelah berhasil menghubungkan, ucapkan terima kasih dan akhiri percakapan dengan sopan.
"""
# sales_tools_only is now a global variable defined in yhqdZOjBc8B5

global sales_team_agent # Declare global to modify the shared variable
sales_team_agent = _create_agent_instance(
    current_model_instance=model, # Use the globally available 'model' instance
    tools_list=sales_tools_only,
    system_prompt_text=sales_system_instruction,
)
print("Sales Team Agent created successfully.")

Sales Team Agent created successfully.


In [ ]:
# --- Recruitment Team Agent ---
recruitment_system_instruction = """
Anda adalah Recruitment Assistant dari perusahaan IT. Tugas Anda adalah memberikan informasi tentang lowongan kerja, persyaratan, proses seleksi, dan status lamaran. Anda harus fokus pada posisi seperti Laravel Developer, Fullstack Developer, Mobile Developer (Flutter), UI/UX Designer, QA Engineer, DevOps Engineer, Project Manager, Business Analyst, Sales Executive, dan HR Officer. Selalu tanyakan apakah pengguna ingin dihubungkan ke tim Recruitment. Jika pengguna secara eksplisit meminta untuk berbicara dengan tim Recruitment atau informasi lebih lanjut tentang lowongan, Anda HARUS menggunakan tool `connect_to_recruitment_team`. Setelah berhasil menghubungkan, ucapkan terima kasih dan akhiri percakapan dengan sopan.
"""
# recruitment_tools_only is now a global variable defined in yhqdZOjBc8B5

global recruitment_team_agent # Declare global to modify the shared variable
recruitment_team_agent = _create_agent_instance(
    current_model_instance=model, # Use the globally available 'model' instance
    tools_list=recruitment_tools_only,
    system_prompt_text=recruitment_system_instruction,
)
print("Recruitment Team Agent created successfully.")

Recruitment Team Agent created successfully.


In [ ]:
import re
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage

def chat_with_hr():
    thread_id = "1"
    config = {"configurable": {"thread_id": thread_id}}

    print("Welcome to HR Bot! Type 'q' to quit, 'kembali ke HR' to return to main HR.")

    current_agent = barista_agent # Start with the main HR agent
    agent_name = "HRBot"

    while True:
        user_input = input(f"You ({agent_name}): ")

        if user_input.lower() in ['q', 'quit', 'exit']:
            print(f"{agent_name}: Thank you for visiting! Have a great day!")
            break

        if user_input.lower() == 'kembali ke hr':
            current_agent = barista_agent
            agent_name = "HRBot"
            print(f"{agent_name}: Anda telah kembali ke agen HR utama. Ada yang bisa saya bantu lagi?")
            continue

        try:
            response = current_agent.invoke(
                {
                    "messages": [
                        {
                            "role": "user",
                            "content": user_input
                        }
                    ]
                },
                config=config
            )

            # Default display content to the final AI message content
            display_content = response["messages"][-1].content
            original_agent_final_message_content = display_content # Keep track of the final AI message for comparison

            # Check for agent handoff signals by iterating through all messages,
            # specifically looking for ToolMessage content if a tool was called.
            tool_output_for_handoff = None
            if current_agent == barista_agent: # Only main HRBot can initiate handoff
                for msg in response["messages"]:
                    if isinstance(msg, ToolMessage):
                        if "[AGENT_SALES]" in msg.content:
                            current_agent = sales_team_agent
                            agent_name = "Sales Team Agent"
                            print("\n--- Switched to Sales Team Agent ---")
                            tool_output_for_handoff = msg.content
                            break # Found handoff, no need to check further messages
                        elif "[AGENT_RECRUITMENT]" in msg.content:
                            current_agent = recruitment_team_agent
                            agent_name = "Recruitment Team Agent"
                            print("\n--- Switched to Recruitment Team Agent ---")
                            tool_output_for_handoff = msg.content
                            break # Found handoff, no need to check further messages

            if tool_output_for_handoff:
                # If handoff happened based on tool output, use that for display
                # and strip the handoff tags.
                display_content = tool_output_for_handoff.replace("[AGENT_SALES]", "").replace("[AGENT_RECRUITMENT]", "").strip()
            else:
                # Otherwise, use the original final AI message content as set initially
                display_content = original_agent_final_message_content

            # Strip markdown (e.g., asterisks) for plain text output after handoff check
            display_content = re.sub(r'[*#]', '', display_content).strip()
            print(f"{agent_name}: {display_content}")

        except Exception as e:
            print(f"An error occurred: {e}")
            print("Please try again or type 'q' to quit.")

In [ ]:
chat_with_hr()

Welcome to HR Bot! Type 'q' to quit, 'kembali ke HR' to return to main HR.
You (HRBot): halo
An error occurred: Error calling model 'gemini-2.5-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash-lite\nPlease retry in 25.29083951s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/gene

In [ ]:
pip install -q streamlit pyngrok

Next, I will create the `app.py` file containing the Streamlit application code. This code will integrate your existing chatbot logic with a Streamlit UI, using session state to manage the conversation.

In [ ]:
%%writefile app.py
import streamlit as st
import re
import os
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_google_genai import ChatGoogleGenerativeAI

# --- Configuration for Agent Initialization ---

# Get API key from environment variable (set by the Colab notebook)
try:
    GEMINI_API_KEY = os.environ.get('GOOGLE_API_KEY') # Expected to be set as GOOGLE_API_KEY
    if not GEMINI_API_KEY:
        raise ValueError("GOOGLE_API_KEY environment variable is empty or not set.")
    print("GEMINI API Key retrieved from environment.")
except Exception as e:
    st.error(
        f"Error loading GEMINI API Key: {e}. \n\n"
        "Please ensure your 'GEMINI' secret is set in Colab and passed as 'GOOGLE_API_KEY' environment variable when launching Streamlit.\n"
        "1. Click the \U0001F511 'Secrets' icon on the left sidebar of your Colab notebook.\n"
        "2. Add a new secret with the exact name 'GEMINI'.\n"
        "3. Paste your Google AI Studio API key into the 'Value' field.\n"
        "4. Make sure to enable 'Notebook access' for this secret.\n"
        "5. Re-run the cell that generates `app.py` (cell `1fdfd0b1`).\n"
        "6. Re-run the cell that launches Streamlit (cell `7cf6199f`)."
    )
    st.stop() # Stop Streamlit app if API key is not available

# Set the GOOGLE_API_KEY environment variable for the model if not already set
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

GEMINI_MODELS = ["gemini-2.5-flash-lite", "gemini-1.5-flash", "gemini-1.5-pro"]
_current_gemini_model_index = 0 # Use the first model for simplicity

# Global instances (within app.py scope)
checkpointer = InMemorySaver()
model = ChatGoogleGenerativeAI(
    model=GEMINI_MODELS[_current_gemini_model_index],
    temperature=0.2
)

# --- Tool Definitions ---
def connect_to_sales_team() -> str:
    """Connects the user to the sales team for product demos and pricing.
    This tool should be used when the user explicitly expresses interest in speaking with the sales team or getting a demo/quote."""
    return "Anda telah berhasil dihubungkan dengan tim Sales. Mereka akan segera menghubungi Anda. [AGENT_SALES]"

def connect_to_recruitment_team() -> str:
    """Connects the user to the recruitment team for job applications or inquiries.
    This tool should be used when the user explicitly expresses interest in speaking with the recruitment team or getting more information about job openings."""
    return "Anda telah berhasil dihubungkan dengan tim Recruitment. Mereka akan segera menghubungi Anda. [AGENT_RECRUITMENT]"

hr_tools = [connect_to_sales_team, connect_to_recruitment_team]
sales_tools_only = [connect_to_sales_team]
recruitment_tools_only = [connect_to_recruitment_team]

# --- System Instructions ---
system_instruction = """
Anda adalah HR Assistant Bot, asisten virtual resmi perusahaan IT yang bertugas membantu calon klien, calon karyawan, dan pengunjung perusahaan.

Perusahaan kami bergerak di bidang Pengembangan Software, Implementasi Sistem ERP, HRIS, Mobile Application, Integrasi API, Infrastruktur Server, dan Konsultasi Teknologi Informasi.

Anda memiliki dua peran utama:

1. SALES CONSULTANT
   Jika pengguna bertanya tentang produk atau layanan perusahaan, Anda akan menjelaskan detail produk, manfaat, fitur, dan proses implementasinya.

2. RECRUITMENT ASSISTANT
   Jika pengguna bertanya tentang lowongan kerja, persyaratan, proses seleksi, atau status lamaran, Anda akan memberikan informasi yang relevan mengenai rekrutmen.

==================================================
PROFIL PERUSAHAAN
==================================================

Kami adalah perusahaan IT yang menyediakan solusi teknologi untuk membantu bisnis menjadi lebih efisien, terintegrasi, dan scalable.

Layanan utama perusahaan meliputi:

1. ERP SYSTEM DEVELOPMENT
   - Enterprise Resource Planning (ERP)
   - Modul Finance & Accounting
   - Purchasing
   - Inventory & Warehouse
   - Sales & CRM
   - Project Management
   - Asset Management

2. HRIS (Human Resource Information System)
   - Absensi online
   - Pengajuan cuti
   - Overtime
   - Payroll
   - Shift management
   - Employee self service

3. CUSTOM SOFTWARE DEVELOPMENT
   - Web application
   - Mobile application Android/iOS
   - Dashboard & reporting
   - API integrations

4. AI & CHATBOT SOLUTIONS
   - Chatbot WhatsApp
   - Customer service automation
   - FAQ bot
   - AI knowledge assistant

5. CLOUD & DEVOPS SERVICES
   - VPS setup
   - Docker deployment
   - CI/CD pipeline
   - Server monitoring
   - Backup automation

6. IT CONSULTING
   - Digital transformation
   - System architecture
   - Security review
   - Performance optimization

==================================================
INSTRUKSI UMUM
==================================================

- Gunakan Bahasa Indonesia yang profesional, ramah, dan mudah dipahami.
- Fokus hanya pada topik berikut:
  - Produk dan layanan perusahaan.
  - Penawaran harga dan proses implementasi.
  - Lowongan kerja dan proses rekrutmen.
- Jangan membahas topik di luar ruang lingkup perusahaan.
- Jika pertanyaan tidak relevan, arahkan kembali ke topik layanan atau rekrutmen.

==================================================
PERAN SALES CONSULTANT
==================================================

Jika pengguna tertarik pada produk atau layanan:

- Jelaskan kebutuhan bisnis yang dapat diselesaikan.
- Jelaskan fitur utama produk.
- Jelaskan estimasi waktu implementasi.
- Jelaskan manfaat bisnis.
- Tawarkan untuk menghubungkan pengguna dengan tim sales.

Contoh layanan yang dapat dijelaskan:
- ERP untuk perusahaan outsourcing.
- HRIS untuk absensi dan payroll.
- Sistem recruitment.
- CRM.
- Chatbot WhatsApp AI.
- Integrasi dengan Odoo, SAP, dan API pihak ketiga.

Di akhir penjelasan, selalu tawarkan:
"Apakah Bapak/Ibu ingin berbicara langsung dengan tim Sales kami untuk mendapatkan demo dan penawaran harga?"

**PENTING SEKALI:** Jika pengguna secara eksplisit meminta untuk berbicara dengan tim Sales atau meminta demo/penawaran harga, atau bahkan hanya menyebutkan kata 'Sales', 'Tim Sales', 'Hubungkan saya dengan Sales', Anda HARUS menggunakan tool `connect_to_sales_team` secara langsung. Segera setelah menggunakan tool ini, respons Anda WAJIB DAN MUTLAK berupa pesan konfirmasi yang S-E-L-A-L-U diakhiri dengan tag `[AGENT_SALES]`. Pastikan tag ini adalah bagian TERAKHIR dari *seluruh* respons Anda, tanpa ada teks, spasi, atau karakter lain setelahnya. Contoh respons yang valid: 'Anda telah berhasil dihubungkan dengan tim Sales. Mereka akan segera menghubungi Anda. [AGENT_SALES]'

==================================================
PERAN RECRUITMENT ASSISTANT
==================================================

Jika pengguna bertanya tentang lowongan kerja, Anda dapat menjelaskan posisi berikut:

1. Laravel Developer
2. Fullstack Developer
3. Mobile Developer (Flutter)
4. UI/UX Designer
5. QA Engineer
6. DevOps Engineer
7. Project Manager
8. Business Analyst
9. Sales Executive
10. HR Officer

Informasi yang dapat diberikan:
- Deskripsi pekerjaan.
- Kualifikasi.
- Lokasi kerja.
- Status WFO/Hybrid/Remote.
- Rentang gaji (jika tersedia).
- Tahapan seleksi.

Tahapan seleksi:
1. Screening CV
2. Technical Test
3. Interview HR
4. Interview User
5. Offering Letter

Di akhir penjelasan, selalu tanyakan:
"Apakah Anda ingin saya bantu menghubungkan ke tim Recruitment atau memberikan informasi lowongan yang tersedia?"

**PENTING SEKALI:** Jika pengguna secara eksplisit meminta untuk berbicara dengan tim Recruitment atau informasi lebih lanjut tentang lowongan, atau bahkan hanya menyebutkan kata 'Recruitment', 'Tim Recruitment', 'Hubungkan saya dengan Recruitment', Anda HARUS menggunakan tool `connect_to_recruitment_team` secara langsung. Segera setelah menggunakan tool ini, respons Anda WAJIB DAN MUTLAK berupa pesan konfirmasi yang S-E-L-A-L-U diakhiri dengan tag `[AGENT_RECRUITMENT]`. Pastikan tag ini adalah bagian TERAKHIR dari *seluruh* respons Anda, tanpa ada teks, spasi, atau karakter lain setelahnya. Contoh respons yang valid: 'Anda telah berhasil dihubungkan dengan tim Recruitment. Mereka akan segera menghubungi Anda. [AGENT_RECRUITMENT]'

==================================================
ALUR PERCAKAPAN
==================================================

1. Sambut pengguna dengan ramah.
2. Identifikasi apakah pengguna membutuhkan:
   - Informasi produk/layanan (Sales)
   - Informasi lowongan kerja (Recruitment)
3. Berikan jawaban yang sesuai.
4. Tawarkan untuk menghubungkan ke tim terkait.
5. Jika pengguna ingin berbicara dengan tim manusia, arahkan ke:
   - Sales
   - Recruitment
   - Customer Service

==================================================
CONTOH SAPAAN AWAL
==================================================

"Halo, selamat datang di perusahaan kami.

Saya adalah HR Assistant Bot yang dapat membantu Anda terkait:

1. Informasi produk dan layanan IT perusahaan.
2. Demo aplikasi ERP, HRIS, dan chatbot WhatsApp.
3. Informasi lowongan kerja dan proses rekrutmen.

Silakan pilih kebutuhan Anda:
- Sales
- Recruitment"

==================================================
BATASAN
==================================================

- Hanya jawab pertanyaan seputar layanan perusahaan dan rekrutmen.
- Jangan memberikan informasi yang tidak tersedia.
- Jika data tidak diketahui, sarankan untuk menghubungi tim terkait.
- Selalu gunakan bahasa yang sopan dan profesional.

==================================================
PENUTUP
==================================================

Jika pengguna ingin berbicara langsung dengan tim manusia:

- Untuk penawaran produk: arahkan ke tim Sales. **(Pastikan untuk menggunakan tool `connect_to_sales_team` dan respons akhir WAJIB berupa pesan konfirmasi yang diakhiri dengan `[AGENT_SALES]`)**
- Untuk lowongan kerja: arahkan ke tim Recruitment. **(Pastikan untuk menggunakan tool `connect_to_recruitment_team` dan respons akhir WAJIB berupa pesan konfirmasi yang diakhiri dengan `[AGENT_RECRUITMENT]`)**
- Untuk bantuan umum: arahkan ke Customer Service.

Setelah memberikan kontak tim terkait, ucapkan terima kasih dan akhiri percakapan dengan sopan.
"""

sales_system_instruction = """
Anda adalah Sales Consultant dari perusahaan IT yang menyediakan solusi teknologi. Tugas Anda adalah menjelaskan produk, manfaat, fitur, dan proses implementasi kepada calon klien. Anda harus fokus pada layanan perusahaan kami: Pengembangan Software, Implementasi Sistem ERP, HRIS, Mobile Application, Integrasi API, Infrastruktur Server, dan Konsultasi Teknologi Informasi. Selalu tawarkan untuk menghubungkan pengguna dengan tim Sales kami. Jika pengguna secara eksplisit meminta untuk berbicara dengan tim Sales atau meminta demo/penawaran harga, Anda HARUS menggunakan tool `connect_to_sales_team`. Setelah berhasil menghubungkan, ucapkan terima kasih dan akhiri percakapan dengan sopan.
"""

recruitment_system_instruction = """
Anda adalah Recruitment Assistant dari perusahaan IT. Tugas Anda adalah memberikan informasi tentang lowongan kerja, persyaratan, proses seleksi, dan status lamaran. Anda harus fokus pada posisi seperti Laravel Developer, Fullstack Developer, Mobile Developer (Flutter), UI/UX Designer, QA Engineer, DevOps Engineer, Project Manager, Business Analyst, Sales Executive, dan HR Officer. Selalu tanyakan apakah pengguna ingin dihubungkan ke tim Recruitment. Jika pengguna secara eksplisit meminta untuk berbicara dengan tim Recruitment atau informasi lebih lanjut tentang lowongan, Anda HARUS menggunakan tool `connect_to_recruitment_team`. Setelah berhasil menghubungkan, ucapkan terima kasih dan akhiri percakapan dengan sopan.
"""

# Helper function to create a single agent instance
def _create_agent_instance(current_model_instance, tools_list, system_prompt_text):
    return create_agent(
        model=current_model_instance,
        tools=tools_list,
        system_prompt=system_prompt_text,
        checkpointer=checkpointer
    )

# --- Agent Initialization ---
barista_agent = _create_agent_instance(
    current_model_instance=model,
    tools_list=hr_tools,
    system_prompt_text=system_instruction
)

sales_team_agent = _create_agent_instance(
    current_model_instance=model,
    tools_list=sales_tools_only,
    system_prompt_text=sales_system_instruction,
)

recruitment_team_agent = _create_agent_instance(
    current_model_instance=model,
    tools_list=recruitment_tools_only,
    system_prompt_text=recruitment_system_instruction,
)

# Map agent names to instances for session state management
agent_instances = {
    'HRBot': barista_agent,
    'Sales Team Agent': sales_team_agent,
    'Recruitment Team Agent': recruitment_team_agent
}

# Streamlit UI setup
st.set_page_config(page_title="HR Chatbot", page_icon="🤖")
st.title("🤖 HR Chatbot")

st.markdown("""
<style>
body {
    font-family: Arial, sans-serif;
}
.main .block-container {
    padding-top: 2rem; /* Adjust as needed */
    padding-bottom: 2rem;
}
.st-chat-message-container {
    background-color: #f0f2f5; /* Light grey background for the chat area */
    border-radius: 8px;
    padding: 10px;
    margin-bottom: 15px;
}
.stChatMessage {
    margin-bottom: 8px;
    display: flex; /* Use flexbox for alignment */
    width: 100%;
}
/* Styling for user messages */
.stChatMessage.st-chat-message-user > div {
    background-color: #dcf8c6; /* WhatsApp green */
    border-radius: 8px 8px 0 8px; /* Rounded top, square bottom-left */
    padding: 8px 12px;
    margin-left: auto; /* Align user messages to the right */
    max-width: 75%; /* Limit message width */
    color: #000;
    box-shadow: 0 1px 0.5px rgba(0, 0, 0, 0.13);
}
/* Styling for assistant messages */
.stChatMessage.st-chat-message-assistant > div {
    background-color: #ffffff; /* White */
    border-radius: 8px 8px 8px 0; /* Rounded top, square bottom-right */
    padding: 8px 12px;
    margin-right: auto; /* Align assistant messages to the left */
    max-width: 75%; /* Limit message width */
    color: #000;
    box-shadow: 0 1px 0.5px rgba(0, 0, 0, 0.13);
}
/* Further adjustments for text within messages if needed */
.stChatMessage p {
    margin: 0; /* Remove default paragraph margins */
    white-space: pre-wrap; /* Preserve whitespace and wrap text */
    word-break: break-word;
}
/* Chat input styling */
.st-chat-input-container {
    background-color: #f0f2f5;
    padding: 10px;
    border-top: 1px solid #e0e0e0;
}
.st-chat-input-container input {
    border-radius: 20px;
    padding: 10px 15px;
    border: none;
    box-shadow: none;
}
.st-chat-input-container button {
    background-color: #075e54; /* WhatsApp send button color */
    color: white;
    border-radius: 50%;
    width: 40px;
    height: 40px;
    display: flex;
    align-items: center;
    justify-content: center;
}
</style>
""", unsafe_allow_html=True)

# Initialize session state for chat history and current agent if not already present
if 'chat_history' not in st.session_state:
    st.session_state['chat_history'] = []
if 'current_agent_name' not in st.session_state:
    st.session_state['current_agent_name'] = 'HRBot'
    st.session_state['current_agent_instance'] = agent_instances['HRBot'] # Initial agent instance

# Display chat history
for role, message in st.session_state['chat_history']:
    with st.chat_message(role):
        st.markdown(message)

# Chat input
user_query = st.chat_input("Type your message here...")

if user_query:
    # Add user message to history and display
    st.session_state['chat_history'].append(("user", user_query))
    with st.chat_message("user"):
        st.markdown(user_query)

    # Handle 'kembali ke HR' command
    if user_query.lower() == 'kembali ke hr':
        st.session_state['current_agent_name'] = 'HRBot'
        st.session_state['current_agent_instance'] = agent_instances['HRBot'] # Reset to main HR agent
        response_message = "Anda telah kembali ke agen HR utama. Ada yang bisa saya bantu lagi?"
        st.session_state['chat_history'].append(("assistant", response_message))
        with st.chat_message("assistant"):
            st.markdown(response_message)
        st.rerun() # Use st.rerun() for newer Streamlit versions
    else:
        try:
            current_agent = st.session_state['current_agent_instance']
            agent_name = st.session_state['current_agent_name']

            # Construct messages for the agent based on history (only current turn for invoke)
            messages_for_agent = [
                HumanMessage(content=user_query)
            ]

            # Langgraph requires a config with thread_id
            config = {"configurable": {"thread_id": "1"}}

            response = current_agent.invoke(
                {
                    "messages": messages_for_agent
                },
                config=config
            )

            display_content = response["messages"][-1].content
            original_agent_final_message_content = display_content

            # Check for agent handoff signals
            tool_output_for_handoff = None
            # Handoff only happens if the main HRBot is the current agent
            if agent_name == 'HRBot':
                for msg in response["messages"]:
                    if isinstance(msg, ToolMessage):
                        if "[AGENT_SALES]" in msg.content:
                            st.session_state['current_agent_name'] = "Sales Team Agent"
                            st.session_state['current_agent_instance'] = agent_instances['Sales Team Agent']
                            tool_output_for_handoff = msg.content
                            break
                        elif "[AGENT_RECRUITMENT]" in msg.content:
                            st.session_state['current_agent_name'] = "Recruitment Team Agent"
                            st.session_state['current_agent_instance'] = agent_instances['Recruitment Team Agent']
                            tool_output_for_handoff = msg.content
                            break

            if tool_output_for_handoff:
                response_message = tool_output_for_handoff.replace("[AGENT_SALES]", "").replace("[AGENT_RECRUITMENT]", "").strip()
                st.info(f"Switched to {st.session_state['current_agent_name']}!")
            else:
                response_message = original_agent_final_message_content

            response_message = re.sub(r'[*#]', '', response_message).strip()
            st.session_state['chat_history'].append(("assistant", response_message))
            with st.chat_message("assistant"):
                st.markdown(response_message)

            # If agent switched, rerun to update the agent name in the UI prompt
            if tool_output_for_handoff:
                st.rerun()

        except Exception as e:
            error_message = f"An error occurred: {e}"
            st.error(error_message)
            st.session_state['chat_history'].append(("assistant", error_message))

Overwriting app.py


Finally, I will set up ngrok and run the Streamlit application. A public URL will be provided which you can open in your browser to interact with the chatbot.

In [1]:
from pyngrok import ngrok
from google.colab import userdata
import os
import time

# Ensure all necessary packages are installed for the streamlit environment
# Adding this here ensures streamlit picks them up, especially for nohup.
!pip install -qU langgraph==1.1.5 langchain-google-genai streamlit pyngrok

# Terminate any previous ngrok tunnels
ngrok.kill()

# Get your ngrok authtoken securely from Colab's secrets manager.
# It is CRUCIAL that the secret is correctly set in Colab's secrets manager
# (click the 🔑 icon on the left sidebar) and named exactly 'NGROK_AUTH_TOKEN'.
try:
    NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
    if not NGROK_AUTH_TOKEN:
        raise ValueError("NGROK_AUTH_TOKEN is empty. Please ensure it's correctly set in Colab Secrets.")
    print("NGROK_AUTH_TOKEN retrieved successfully from Colab secrets.")
except Exception as e:
    raise ValueError(
        "NGROK_AUTH_TOKEN not found in Colab Secrets. "
        "Please follow these steps:\n"
        "1. Click the 🔑 'Secrets' icon on the left sidebar of your Colab notebook.\n"
        "2. Add a new secret with the name 'NGROK_AUTH_TOKEN'.\n"
        "3. Paste your ngrok authtoken into the 'Value' field.\n"
        "4. Make sure to enable 'Notebook access' for this secret.\n"
        "5. Rerun this cell."
    ) from e

ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print("ngrok authentication token set.")

# Get GEMINI API key from Colab's secrets manager for passing to Streamlit
try:
    GEMINI_API_KEY = userdata.get('GEMINI')
    if not GEMINI_API_KEY:
        raise ValueError("GEMINI secret is empty or not set. Please ensure it's correctly set in Colab Secrets.")
    print("GEMINI API Key retrieved successfully from Colab secrets.")
except Exception as e:
    raise ValueError(
        "GEMINI API Key not found in Colab Secrets. "
        "Please follow these steps:\n"
        "1. Click the 🔑 'Secrets' icon on the left sidebar of your Colab notebook.\n"
        "2. Add a new secret with the name 'GEMINI'.\n"
        "3. Paste your Google AI Studio API key into the 'Value' field.\n"
        "4. Make sure to enable 'Notebook access' for this secret.\n"
        "5. Re-run cell `1fdfd0b1` (to regenerate `app.py`) and then this cell."
    ) from e

# Kill any existing streamlit processes on port 8501 to ensure a clean restart
!fuser -k 8501/tcp || true

# Export the GEMINI API key as GOOGLE_API_KEY for the shell session
# This ensures nohup process inherits it.
%env GOOGLE_API_KEY=$GEMINI_API_KEY

# Start Streamlit in the background
!nohup streamlit run app.py &

# Create a public URL for the app
# Use a loop to wait for the ngrok tunnel to be established
print("Waiting for ngrok tunnel to establish...")
public_url = None
for _ in range(10): # Try for up to 20 seconds
    try:
        public_url = ngrok.connect(8501)
        if public_url:
            break
    except Exception as e:
        # Suppress excessive retry messages, keep silent retries
        time.sleep(2)

if public_url:
    print(f"Streamlit App URL: {public_url}")
else:
    raise RuntimeError("Failed to establish ngrok tunnel after multiple attempts. "
                       "Please check ngrok logs (e.g., `!cat nohup.out` or `!ngrok logs`) "
                       "and ensure port 8501 is not blocked.")

# To stop ngrok and Streamlit, you might need to manually stop the cell or use `ngrok.kill()`
# in a new cell if the notebook session persists and you want to restart the tunnel.

ModuleNotFoundError: No module named 'pyngrok'